# tabctx quickstart

Fit a table once, keep it as a cached *context* on a server, and predict against it with a scikit-learn-style API. This notebook runs the Python quickstart end to end: install, start a server (CPU is fine), classify, regress with prediction quantiles, and manage contexts. tabctx needs Python 3.12 or newer.

## 1. Install

In [1]:
%pip install -q "tabctx[standalone,tabicl,client]"

Note: you may need to restart the kernel to use updated packages.


## 2. Start a server

The server hosts the TabICL model. The first start downloads the TabICLv2 weights (BSD-3, no account needed). We start it in the background on a free port and wait until it answers. Its log goes to `tabctx-server.log`; if it exits, the cell shows the end of the log.

In [2]:
import atexit
import os
import socket
import subprocess
import sys
import time

from tabctx import TabctxClient

# A free port, so nothing else on this machine (or an earlier run) answers for us.
with socket.socket() as s:
    s.bind(("127.0.0.1", 0))
    port = s.getsockname()[1]

log = open("tabctx-server.log", "w")  # noqa: SIM115 (open while the server runs)
server = subprocess.Popen(
    [sys.executable, "-m", "tabctx.serve", "--port", str(port)],
    # Compiling the model only pays off on large tables; skip it for a fast start.
    env={**os.environ, "TABCTX_BACKEND": "tabicl", "TABCTX_COMPILE": "0"},
    stdout=log,
    stderr=subprocess.STDOUT,
    start_new_session=True,  # interrupting a cell leaves the server running
)
atexit.register(server.terminate)


def log_tail(n=3000):
    with open("tabctx-server.log") as f:
        return f.read()[-n:]


client = TabctxClient(f"http://127.0.0.1:{port}")
deadline = time.monotonic() + 15 * 60
while True:
    if server.poll() is not None:
        raise RuntimeError(f"server exited ({server.returncode}):\n{log_tail()}")
    try:
        print("serving:", client.tabular_models())
        break
    except OSError:  # not listening yet
        if time.monotonic() > deadline:
            raise TimeoutError(f"server not up after 15 min:\n{log_tail()}") from None
        time.sleep(2)

serving: ['tabicl-v2']


## 3. Fit once, predict many (classification)

`fit` sends the table to the server, which encodes it into a cached context. Every later `predict` / `predict_proba` reuses that context.

In [3]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

from tabctx import TabctxClassifier

X, y = fetch_openml("credit-g", version=1, as_frame=True, return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

clf = TabctxClassifier(client=client)
clf.fit(X_train, y_train)
print("context id:", clf.dataset_id_)

context id: sklearn-93906881b2b76a20dd1a638b8bf37862


In [4]:
print("accuracy:", clf.score(X_test, y_test))
clf.predict_proba(X_test[:5])

accuracy: 0.772


array([[0.7023347 , 0.2976653 ],
       [0.05054444, 0.94945556],
       [0.04760655, 0.95239347],
       [0.26296899, 0.73703098],
       [0.39968073, 0.60031927]])

## 4. Regression with prediction quantiles

`TabctxRegressor.predict_quantiles` returns predictive quantiles for each row, not just a point estimate. (The diabetes dataset ships with scikit-learn, so this cell needs no download.)

In [5]:
from sklearn.datasets import load_diabetes

from tabctx import TabctxRegressor

Xr, yr = load_diabetes(as_frame=True, return_X_y=True)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yr, random_state=0)

reg = TabctxRegressor(client=client)
reg.fit(Xr_train, yr_train)
print("context id:", reg.dataset_id_)
reg.predict(Xr_test[:5])

context id: sklearn-4dde377673132b8ee2eb271e49ab7f19


array([255.16026306, 253.02041626, 154.43307495, 117.87319183,
       171.14213562])

In [6]:
import pandas as pd

quantiles = (0.05, 0.5, 0.95)
q = reg.predict_quantiles(Xr_test, quantiles=quantiles)
intervals = pd.DataFrame(
    q, columns=[f"q{q_ * 100:g}" for q_ in quantiles], index=Xr_test.index
)

# Check the intervals on held-out data before relying on them.
inside = yr_test.between(intervals["q5"], intervals["q95"]).mean()
print(f"held-out rows inside the 90% interval: {inside:.0%}")
intervals.head()

held-out rows inside the 90% interval: 92%


,q5,q50,q95
362,147.321915,262.817993,334.048401
249,152.392807,259.652557,327.623749
271,62.779804,146.852875,264.721466
435,42.654739,111.133316,213.519043
400,61.353336,172.986618,275.774841


## 5. List and delete contexts

Contexts live on the server until they are evicted or deleted. `client.datasets()` lists them; `client.delete(...)` removes one. A fitted estimator keeps its training table, so if its context is gone, its next predict quietly fits it again; to remove your data from the server, delete the context and discard the estimator.

In [7]:
[d["dataset_id"] for d in client.datasets()]

['sklearn-4dde377673132b8ee2eb271e49ab7f19',
 'sklearn-93906881b2b76a20dd1a638b8bf37862']

In [8]:
client.delete(clf.dataset_id_)
client.delete(reg.dataset_id_)
client.datasets()

[]

## 6. Shut the server down

In [9]:
server.terminate()
server.wait()
log.close()